#### Day 4: Feature Selection and Data Leakage Prevention

#### Objective
Select suitable features for predicting hotel booking cancellations while preventing data leakage.

In [1]:
import pandas as pd

df = pd.read_csv('../data/hotel_bookings_cleaned.csv')

df.shape

(119208, 33)

Our target column will be is_canceled, before selecting features, we will inspect the availble cols and decide whch ones are more appropriate for predicting cancellation at the time of booking.

#### Examining the dataset cols

In [3]:
df.columns.tolist()

#tolist is used to convert them into python lst

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'reservation_status',
 'reservation_status_date',
 'total_guests',
 'has_agent',
 'has_company']

#### Identifying and removing the data leakge cols

data leakage happens when our model uses info that would not be availble at the time of predictionm

In [6]:
leakage_columns = [
    'reservation_status',
    'reservation_status_date',
    'assigned_room_type',
    'booking_changes',
    'days_in_waiting_list'
]

df_model = df.drop(columns=leakage_columns)

df_model.shape

#IN LEAKAGE_COLS it has the cols to be excluded and then it is dropped
#df_model stores the resulting dframe

(119208, 28)

#### Seperate features x and target y

In [7]:
X = df_model.drop(columns=['is_canceled'])

y = df_model['is_canceled']

print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (119208, 27)
Target shape: (119208,)


#### Identifying numerical and categorical features

What are numerical features?
Features containing numbers, such as:
- lead_time — Number of days between booking and arrival.
- adults — Number of adults.
- adr — Average daily rate.

What are categorical features?
Features representing categories, such as:
- hotel — City Hotel or Resort Hotel.
- deposit_type — No Deposit, Non Refund, etc.
- meal — Meal plan category.

In [8]:
X.dtypes

hotel                              object
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                 object
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                               object
country                            object
market_segment                     object
distribution_channel               object
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                 object
deposit_type                       object
customer_type                      object
adr                               float64
required_car_parking_spaces         int64
total_of_special_requests         

In [9]:

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Number of categorical columns:", len(categorical_cols))

print("\nNumerical columns:", numerical_cols)
print("Number of numerical columns:", len(numerical_cols))


Categorical columns: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type']
Number of categorical columns: 9

Numerical columns: ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'has_agent', 'has_company']
Number of numerical columns: 18


In [10]:
#checking the country col how many unique code are there

X['country'].nunique()


178

In [11]:
X['country'].value_counts().head(15)
#checking which one appears the most

country
PRT    48482
GBR    12119
FRA    10401
ESP     8560
DEU     7285
ITA     3761
IRL     3374
BEL     2342
BRA     2222
NLD     2103
USA     2093
CHE     1726
CN      1279
AUT     1263
SWE     1020
Name: count, dtype: int64

checking countries that has fewer than 100 bookings

In [13]:
country_counts = X['country'].value_counts()

rare_countries = country_counts[country_counts < 100]

print("Countries with fewer than 100 bookings:", len(rare_countries))
print("Bookings from these countries:", rare_countries.sum())


Countries with fewer than 100 bookings: 139
Bookings from these countries: 2387


Decision: keeping the country and handle rare categories inside our preprocessing pipeline, using training-data frequencies to prevent leakage.

In [14]:
print(y.value_counts())

print("\nCancellation percentages:")
print(y.value_counts(normalize=True) * 100)



is_canceled
0    75010
1    44198
Name: count, dtype: int64

Cancellation percentages:
is_canceled
0    62.923629
1    37.076371
Name: proportion, dtype: float64


#### Investigate duplicate feature records
We retained 31,997 duplicate rows during cleaning because identical records may represent different real bookings.

But there's an important ML problem:

If identical bookings appear in both training and testing sets, our model might appear more accurate than it really is.

Before deciding how to split, let's check whether identical feature values sometimes have different cancellation outcomes.

In [16]:

feature_groups = df_model.groupby(
    X.columns.tolist(),
    dropna=False
)['is_canceled'].nunique()

print("Unique feature combinations:", len(feature_groups))
print("Feature combinations with conflicting outcomes:",
      (feature_groups > 1).sum())

#groupby(X.columns.tolist()) groups bookings with identical input features.
#['is_canceled'].nunique() counts how many different cancellation outcomes each group has.
#(feature_groups > 1).sum() counts feature combinations appearing with both canceled and non-canceled outcomes.


Unique feature combinations: 84845
Feature combinations with conflicting outcomes: 304


the above op means

Imagine two hotel bookings with identical information:
- Booking A: City Hotel, 120-day lead time, No Deposit → Canceled
- Booking B: City Hotel, 120-day lead time, No Deposit → Not canceled
Even though their recorded features are identical, their outcomes differ.

Our results show that 304 distinct feature combinations have both cancellation outcomes.

This isn't necessarily a data error. Different customers can make similar bookings and behave differently.

#### Create group IDs
Before splitting, let's assign the same group ID to bookings with identical features.

In [20]:

group_ids = X.groupby(
    X.columns.tolist(),
    dropna=False,
    sort=False
).ngroup()

print("Total bookings:", len(group_ids))
print("Unique groups:", group_ids.nunique())

#assignng each group with numerical id -> ngroup()

Total bookings: 119208
Unique groups: 84845


#### Split the dataset into training and testing sets

using GroupShuffleSplit from scikit-learn.
Why GroupShuffleSplit?
A normal random split might place identical bookings in both training and testing sets.

GroupShuffleSplit keeps all bookings belonging to the same group together.

We'll use:
- 80% training and 20% testing as our target split.
- random_state=42 so we can reproduce the split.
- group_ids to prevent identical feature combinations from crossing the split.

In [22]:

from sklearn.model_selection import GroupShuffleSplit  # Import group-based splitting method

splitter = GroupShuffleSplit(
    n_splits=1,       # Create one train-test split
    test_size=0.2,    # Use 20% of groups for testing
    random_state=42  # Get the same split every time
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=group_ids)  # Keep identical feature groups together
)

X_train = X.iloc[train_idx]  # Select training features
X_test = X.iloc[test_idx]    # Select testing features

y_train = y.iloc[train_idx]  # Select training target values
y_test = y.iloc[test_idx]    # Select testing target values

print("Training features:", X_train.shape)  
print("Testing features:", X_test.shape)    
print("Training target:", y_train.shape)    
print("Testing target:", y_test.shape)     


Training features: (94987, 27)
Testing features: (24221, 27)
Training target: (94987,)
Testing target: (24221,)


#### Check cancellation percentages

In [23]:

print("Training cancellation percentages:")

print(y_train.value_counts(normalize=True) * 100)  # Percentage of each class in training data

print("\nTesting cancellation percentages:")

print(y_test.value_counts(normalize=True) * 100)  # Percentage of each class in testing data



Training cancellation percentages:
is_canceled
0    63.022308
1    36.977692
Name: proportion, dtype: float64

Testing cancellation percentages:
is_canceled
0    62.536642
1    37.463358
Name: proportion, dtype: float64


Verify there is no group overlap
Now we need to confirm that identical feature combinations haven't been placed in both training and testing sets.

In [24]:

train_groups = set(group_ids.iloc[train_idx])  # Get unique group IDs in training data

test_groups = set(group_ids.iloc[test_idx])  # Get unique group IDs in testing data

common_groups = train_groups.intersection(test_groups)  # Find groups present in both datasets

print("Common groups:", len(common_groups))  # Display the number of overlapping groups


Common groups: 0


We used GroupShuffleSplit instead of a normal random split to keep identical feature combinations together and reduce the risk of overly optimistic evaluation caused by duplicate records. This doesn't eliminate every possible source of data leakage, but it addresses this specific issue.

#### Save the training and testing datasets

In [25]:

import os  # Used to work with folders

os.makedirs("../data", exist_ok=True)  # Ensure the data folder exists

X_train.to_csv("../data/X_train.csv", index=False)  # Save training features

X_test.to_csv("../data/X_test.csv", index=False)  # Save testing features

y_train.to_csv("../data/y_train.csv", index=False)  # Save training target

y_test.to_csv("../data/y_test.csv", index=False)  # Save testing target

print("Training and testing datasets saved successfully!")  # Confirmation message


Training and testing datasets saved successfully!


Saving the datasets allows us to load the exact same training and testing data when we begin model development.


## Day 4: Feature Selection and Data Splitting — Summary

### Feature Selection
- Removed 5 potentially leaky or post-booking features.
- Selected 27 input features and 1 target variable.
- Identified 9 categorical and 18 numerical features.
- Investigated rare country categories for future encoding.

### Duplicate Handling
- Identified 84,845 unique feature combinations.
- Found 304 feature combinations with conflicting outcomes.
- Retained duplicate bookings because no unique booking ID exists.

### Train-Test Split
- Used GroupShuffleSplit to prevent identical feature combinations from appearing in both datasets.
- Training set: 94,987 bookings.
- Testing set: 24,221 bookings.
- Training cancellation rate: 36.98%.
- Testing cancellation rate: 37.46%.
- Verified zero overlapping feature groups.

### Saved Files
- X_train.csv
- X_test.csv
- y_train.csv
- y_test.csv

### Next Steps
- Build preprocessing pipelines for categorical and numerical features.
- Train and compare classification models.
- Evaluate model performance using appropriate metrics.
